In [3]:
import sys, os
sys.path.append(os.path.abspath(".."))

In [4]:
from datasets import load_dataset

dataset = load_dataset("go_emotions")
print(dataset["train"][0])

{'text': "My favourite food is anything I didn't have to cook myself.", 'labels': [27], 'id': 'eebbqej'}


In [5]:
from src.preprocessing import get_tokenizer, tokenize, binarize_labels

In [8]:
def process_labels(example):
    example["labels"] = binarize_labels(example["labels"])
    return example

dataset = dataset.map(process_labels)

In [9]:
tokenizer = get_tokenizer()

In [ ]:
def process_text(example):
    tokenized = tokenize(example["text"], tokenizer)
    example["input_ids"] = tokenized["input_ids"][0]
    example["attention_mask"] = tokenized["attention_mask"][0]
    return example

tokenized_dataset = dataset.map(process_text)

In [11]:
small_train = tokenized_dataset["train"].select(range(10000))
small_val = tokenized_dataset["validation"].select(range(2000))

In [13]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=28,
    problem_type="multi_label_classification"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6110.48it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
)

In [15]:
import torch

def data_collator(features):
    batch = {}

    # input_ids and attention_mask 
    batch["input_ids"] = torch.stack([torch.tensor(f["input_ids"]) for f in features])
    batch["attention_mask"] = torch.stack([torch.tensor(f["attention_mask"]) for f in features])
 
    batch["labels"] = torch.stack([
        torch.tensor(f["labels"], dtype=torch.float32) for f in features
    ])

    return batch

In [16]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator,
)

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.121041,0.119031
2,0.108879,0.101201
3,0.081857,0.098033


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.81s/it]
/Users/jinruzhang/Documents/NLP/group/info7160-goemotions/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.15it/s]
/Users/jinruzhang/Documents/NLP/group/info7160-goemotions/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.07it/s]


TrainOutput(global_step=1875, training_loss=0.11532271253267924, metrics={'train_runtime': 6033.8705, 'train_samples_per_second': 4.972, 'train_steps_per_second': 0.311, 'total_flos': 1973793576960000.0, 'train_loss': 0.11532271253267924, 'epoch': 3.0})

In [17]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    "results/checkpoint-1250"
)

tokenizer = AutoTokenizer.from_pretrained(
    "results/checkpoint-1250"
)

trainer = Trainer(model=model)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5903.09it/s]


In [19]:
tokenized_dataset = tokenized_dataset.map(
    lambda x: {"labels": [float(i) for i in x["labels"]]}
)

Map: 100%|██████████| 5427/5427 [00:00<00:00, 23010.19 examples/s]


In [21]:
test_dataset = tokenized_dataset["test"].remove_columns("labels")
predictions = trainer.predict(test_dataset)

In [25]:
import numpy as np

logits = predictions.predictions

# sigmoid
probs = 1 / (1 + np.exp(-logits))

# threshold 0.3
pred_labels = (probs > 0.3).astype(int)

In [27]:
import os
import json
 
os.makedirs("outputs", exist_ok=True)

output_file = "../outputs/bert_preds.jsonl"

with open(output_file, "w") as f:
    for i, example in enumerate(tokenized_dataset["test"]):
        record = {
            "id": example["id"],
            "text": example["text"],
            "true_labels": example["labels"],
            "pred_labels": pred_labels[i].tolist()
        }
        f.write(json.dumps(record) + "\n")

print("saved to", output_file)

saved to ../outputs/bert_preds.jsonl


In [28]:
model.save_pretrained("outputs/bert_checkpoint")
tokenizer.save_pretrained("outputs/bert_checkpoint")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.45s/it]


('outputs/bert_checkpoint/tokenizer_config.json',
 'outputs/bert_checkpoint/tokenizer.json')